In [ ]:

import numpy as np
from scipy.stats import norm
from scipy.stats import qmc
from scipy.stats.qmc import Sobol
import yfinance as yf

# Parameters
spot_price = 602.81
strike_price = 610
maturity = 3/252  #in years
risk_free_rate = 0.045  # Annual risk-free rate
volatility = 0.14 # Annual volatility (14%)
n_simulations = 8192# Number of Quasi-Monte Carlo simulations
n_steps = 100  # Number of time steps
seed = 40  # Random seed for reproducibility
dt = maturity / n_steps  # Time step size

# Quasi-Monte Carlo: Sobol sequence generation
def generate_qmc_price_paths(spot_price, risk_free_rate, volatility, maturity, n_simulations, n_steps, seed):
    dt = maturity / n_steps
    sampler = qmc.Sobol(d=n_steps, scramble=True, seed=seed)
    sobol_sequences = sampler.random_base2(m=int(np.log2(n_simulations)))

    price_paths = np.zeros((n_simulations, n_steps + 1))
    price_paths[:, 0] = spot_price

    for t in range(1, n_steps + 1):
        z = norm.ppf(sobol_sequences[:, t - 1])
        price_paths[:, t] = price_paths[:, t - 1] * np.exp(
            (risk_free_rate - 0.5 * volatility**2) * dt + volatility * np.sqrt(dt) * z
        )
    return price_paths

# Least Squares Monte Carlo (LSM) for American option pricing
def lsm_american_option(price_paths, strike, rf_rate, dt, option_type="call"):
    n_simulations, n_steps = price_paths.shape
    option_values = np.maximum(
        0,
        price_paths[:, -1] - strike if option_type == "call" else strike - price_paths[:, -1],
    )

    for t in range(n_steps - 1, 0, -1):
        in_the_money = (
            price_paths[:, t] > strike if option_type == "call" else price_paths[:, t] < strike
        )
        itm_paths = price_paths[in_the_money, t]
        itm_values = option_values[in_the_money]

        if len(itm_paths) > 0:
            X = itm_paths
            Y = itm_values * np.exp(-rf_rate * dt)
            regression = np.polyfit(X, Y, 2)
            continuation_values = np.polyval(regression, itm_paths)
            option_values[in_the_money] = np.maximum(itm_values, continuation_values)

    return np.mean(option_values) * np.exp(-rf_rate * dt)

# Greeks Calculation
def calculate_greeks(spot_price, strike_price, maturity, risk_free_rate, volatility, n_simulations, n_steps, seed, epsilon=0.01*spot_price):
    dt = maturity / n_steps

    # Base price
    price_paths = generate_qmc_price_paths(spot_price, risk_free_rate, volatility, maturity, n_simulations, n_steps, seed)
    base_price = lsm_american_option(price_paths, strike_price, risk_free_rate, dt)

    # Delta
    price_paths_up = generate_qmc_price_paths(spot_price + epsilon, risk_free_rate, volatility, maturity, n_simulations, n_steps, seed)
    price_paths_down = generate_qmc_price_paths(spot_price - epsilon, risk_free_rate, volatility, maturity, n_simulations, n_steps, seed)
    price_up = lsm_american_option(price_paths_up, strike_price, risk_free_rate, dt)
    price_down = lsm_american_option(price_paths_down, strike_price, risk_free_rate, dt)
    delta = (price_up - price_down) / (2 * epsilon)

    # Gamma
    gamma = (price_up - 2 * base_price + price_down) / (epsilon**2)

    # Vega
    price_paths_vol_up = generate_qmc_price_paths(spot_price, risk_free_rate, volatility + epsilon, maturity, n_simulations, n_steps, seed)
    price_paths_vol_down = generate_qmc_price_paths(spot_price, risk_free_rate, volatility - epsilon, maturity, n_simulations, n_steps, seed)
    price_vol_up = lsm_american_option(price_paths_vol_up, strike_price, risk_free_rate, dt)
    price_vol_down = lsm_american_option(price_paths_vol_down, strike_price, risk_free_rate, dt)
    vega = (price_vol_up - price_vol_down) / (2 * epsilon)

    # Theta
    price_paths_shorter = generate_qmc_price_paths(spot_price, risk_free_rate, volatility, maturity - epsilon, n_simulations, n_steps, seed)
    price_shorter = lsm_american_option(price_paths_shorter, strike_price, risk_free_rate, dt)
    theta = (price_shorter - base_price) / epsilon

    # Rho
    price_paths_rf_up = generate_qmc_price_paths(spot_price, risk_free_rate + epsilon, volatility, maturity, n_simulations, n_steps, seed)
    price_paths_rf_down = generate_qmc_price_paths(spot_price, risk_free_rate - epsilon, volatility, maturity, n_simulations, n_steps, seed)
    price_rf_up = lsm_american_option(price_paths_rf_up, strike_price, risk_free_rate, dt)
    price_rf_down = lsm_american_option(price_paths_rf_down, strike_price, risk_free_rate, dt)
    rho = (price_rf_up - price_rf_down) / (2 * epsilon)

    return {"Delta": delta, "Gamma": gamma, "Vega": vega, "Theta": theta, "Rho": rho}

# Main Execution
price_paths = generate_qmc_price_paths(spot_price, risk_free_rate, volatility, maturity, n_simulations, n_steps, seed)
qmc_price = lsm_american_option(price_paths, strike_price, risk_free_rate, dt)
#fdm_price = finite_difference_method(spot_price, strike_price, risk_free_rate, volatility, maturity, n_steps,10, dt, option_type="call")
greeks = calculate_greeks(spot_price, strike_price, maturity, risk_free_rate, volatility, n_simulations, n_steps, seed)

print(f"QMC American Option Price: {qmc_price:.4f}")
#print(f"FDM American Option Price: {fdm_price:.4f}")
for greek, value in greeks.items():
    print(f"{greek}: {value:.4f}")


<ipython-input-12-cedd8ad89346>:52: RankWarning: Polyfit may be poorly conditioned
  regression = np.polyfit(X, Y, 2)
<ipython-input-12-cedd8ad89346>:52: RankWarning: Polyfit may be poorly conditioned
  regression = np.polyfit(X, Y, 2)
<ipython-input-12-cedd8ad89346>:52: RankWarning: Polyfit may be poorly conditioned
  regression = np.polyfit(X, Y, 2)
<ipython-input-12-cedd8ad89346>:52: RankWarning: Polyfit may be poorly conditioned
  regression = np.polyfit(X, Y, 2)
<ipython-input-12-cedd8ad89346>:52: RankWarning: Polyfit may be poorly conditioned
  regression = np.polyfit(X, Y, 2)
<ipython-input-12-cedd8ad89346>:30: RuntimeWarning: invalid value encountered in sqrt
  (risk_free_rate - 0.5 * volatility**2) * dt + volatility * np.sqrt(dt) * z


QMC American Option Price: 9.3836
Delta: 2.3906
Gamma: 0.3921
Vega: 15.0981
Theta: nan
Rho: 6.6656


<ipython-input-12-cedd8ad89346>:52: RankWarning: Polyfit may be poorly conditioned
  regression = np.polyfit(X, Y, 2)
<ipython-input-12-cedd8ad89346>:52: RankWarning: Polyfit may be poorly conditioned
  regression = np.polyfit(X, Y, 2)
<ipython-input-12-cedd8ad89346>:52: RankWarning: Polyfit may be poorly conditioned
  regression = np.polyfit(X, Y, 2)
<ipython-input-12-cedd8ad89346>:52: RankWarning: Polyfit may be poorly conditioned
  regression = np.polyfit(X, Y, 2)
<ipython-input-12-cedd8ad89346>:52: RankWarning: Polyfit may be poorly conditioned
  regression = np.polyfit(X, Y, 2)
<ipython-input-12-cedd8ad89346>:52: RankWarning: Polyfit may be poorly conditioned
  regression = np.polyfit(X, Y, 2)
<ipython-input-12-cedd8ad89346>:52: RankWarning: Polyfit may be poorly conditioned
  regression = np.polyfit(X, Y, 2)
<ipython-input-12-cedd8ad89346>:52: RankWarning: Polyfit may be poorly conditioned
  regression = np.polyfit(X, Y, 2)
<ipython-input-12-cedd8ad89346>:52: RankWarning: Polyfit

In [ ]:
n_simulations = 5000
time_steps = [5, 10, 20, 50, 100, 200]  # Different numbers of time steps
prices = []

for n in time_steps:
    price_paths, dt = generate_qmc_price_paths(spot_price, maturity, n, n_simulations, risk_free_rate, volatility,40)
    price = lsm_american_option(price_paths, strike_price, risk_free_rate, dt, option_type)
    prices.append(price)

# Plot the results
plt.figure(figsize=(10, 6))
plt.plot(time_steps, prices, marker='o', label=f'American {option_type.capitalize()} Option Price')
plt.title("Dependence of American Option Price on Time Steps")
plt.xlabel("Number of Time Steps (n)")
plt.ylabel("Option Price")
plt.grid()
plt.legend()
plt.show()

ValueError: d must be a non-negative integer value

In [ ]:
def finite_difference_method(spot, strike, rate, volatility, T, steps, grid_size, dt, option_type="call"):
    dt = dt
    ds = 2 * spot / grid_size
    S = np.linspace(0, 2 * spot, grid_size + 1)
    V = np.zeros((grid_size + 1, steps + 1))

    # Boundary and initial conditions
    if option_type == "call":
        V[:, -1] = np.maximum(S - strike, 0)
        V[0, :] = 0  # At S=0
        V[-1, :] = 2 * spot - strike  # Deep ITM call
    elif option_type == "put":
        V[:, -1] = np.maximum(strike - S, 0)
        V[0, :] = strike  # Deep ITM put
        V[-1, :] = 0  # At S -> infinity

    # Backward induction
    for t in reversed(range(steps)):
        for i in range(1, grid_size):
            delta = (V[i + 1, t + 1] - V[i - 1, t + 1]) / (2 * ds)
            gamma = (V[i + 1, t + 1] - 2 * V[i, t + 1] + V[i - 1, t + 1]) / (ds ** 2)
            theta = -0.5 * volatility ** 2 * S[i] ** 2 * gamma - rate * S[i] * delta + rate * V[i, t + 1]
            V[i, t] = V[i, t + 1] + dt * theta

        # Early exercise for American options
        if option_type == "call":
            V[:, t] = np.maximum(V[:, t], S - strike)
        elif option_type == "put":
            V[:, t] = np.maximum(V[:, t], strike - S)

    # Interpolate to find the price for the given spot
    price = np.interp(spot, S, V[:, 0])
    return price

In [ ]:
def market_option_price(ticker, strike, expiration):
    # Fetch the SPY options chain for a given expiration date
    spy = yf.Ticker(ticker)
    options_chain = spy.option_chain(expiration)

    # Get the calls and puts for the expiration
    calls = options_chain.calls
    puts = options_chain.puts

    # Find the closest strike price
    actual_call_price = calls[calls['strike'] == strike]['lastPrice'].values
    actual_put_price = puts[puts['strike'] == strike]['lastPrice'].values

    # If the option exists, return the price
    if len(actual_call_price) > 0:
        return actual_call_price[0], 'call'
    elif len(actual_put_price) > 0:
        return actual_put_price[0], 'put'
    else:
        raise ValueError("Option with the given strike price not found.")
expiration_date = "2024-12-16"  # Expiration date for SPY options (adjust as needed)
try:
    actual_price, option_type = market_option_price("SPY", strike_price, expiration_date)
    error = abs(qmc_price - actual_price)
    print(f"QMC American Option Price: {qmc_price:.2f}")
    print(f"Actual Option Price: {actual_price:.2f}")
    print(f"Error: {error:.2f}")
except ValueError as e:
    print(e)

NameError: name 'strike_price' is not defined

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats.qmc import Sobol
from sklearn.linear_model import LinearRegression
import datetime
file_path = "/content/yahoo_options-2.csv"
data =  pd.read_csv(file_path)
# Helper function for payoff
def payoff(spot, strike, option_type="call"):
    if option_type == "call":
        return np.maximum(spot - strike, 0)
    elif option_type == "put":
        return np.maximum(strike - spot, 0)
    else:
        raise ValueError("Invalid option type. Use 'call' or 'put'.")

# Fetch SPY data from Yahoo Finance
def fetch_data(ticker="SPY", start_date=None, end_date=None):
    data = yf.download(ticker, start=start_date, end=end_date)
    return data

# Sobol Quasi Monte Carlo paths
def sobol_paths(spot, volatility, rate, T, steps, samples):
    dt = T / steps
    sobol = Sobol(d=steps, scramble=True)
    sobol_samples = sobol.random(samples)
    normals = (sobol_samples - 0.5) * np.sqrt(12)  # Map to standard normal
    increments = (rate - 0.5 * volatility**2) * dt + volatility * np.sqrt(dt) * normals
    log_paths = np.cumsum(increments, axis=1)
    paths = spot * np.exp(log_paths)
    paths = np.hstack((np.full((samples, 1), spot), paths))
    return paths

# Least Squares Monte Carlo (LSM) for American Option Pricing
def least_squares_monte_carlo(spot, strike, rate, volatility, T, steps, samples, option_type="call"):
    paths = sobol_paths(spot, volatility, rate, T, steps, samples)
    dt = T / steps
    discounts = np.exp(-rate * dt * np.arange(1, steps + 1))
    payoffs = payoff(paths, strike, option_type)
    exercise_values = payoffs[:, -1]
    continuation_values = np.zeros_like(exercise_values)

    for t in reversed(range(steps)):
        in_the_money = payoff(paths[:, t], strike, option_type) > 0
        if np.any(in_the_money):
            X = paths[in_the_money, t].reshape(-1, 1)
            Y = discounts[t] * exercise_values[in_the_money]
            model = LinearRegression().fit(X, Y)
            continuation_values[in_the_money] = model.predict(X)
        exercise_values = np.maximum(payoffs[:, t], continuation_values)
    return np.mean(exercise_values) * np.exp(-rate * T)

# Finite Difference Method (FDM)
def finite_difference_method(spot, strike, rate, volatility, T, steps, grid_size, option_type="call"):
    dt = T / steps
    ds = 2 * spot / grid_size
    S = np.linspace(0, 2 * spot, grid_size + 1)
    V = np.zeros((grid_size + 1, steps + 1))

    # Boundary and initial conditions
    if option_type == "call":
        V[:, -1] = np.maximum(S - strike, 0)
        V[0, :] = 0  # At S=0
        V[-1, :] = 2 * spot - strike  # Deep ITM call
    elif option_type == "put":
        V[:, -1] = np.maximum(strike - S, 0)
        V[0, :] = strike  # Deep ITM put
        V[-1, :] = 0  # At S -> infinity

    # Backward induction
    for t in reversed(range(steps)):
        for i in range(1, grid_size):
            delta = (V[i + 1, t + 1] - V[i - 1, t + 1]) / (2 * ds)
            gamma = (V[i + 1, t + 1] - 2 * V[i, t + 1] + V[i - 1, t + 1]) / (ds ** 2)
            theta = -0.5 * volatility ** 2 * S[i] ** 2 * gamma - rate * S[i] * delta + rate * V[i, t + 1]
            V[i, t] = V[i, t + 1] + dt * theta

        # Early exercise for American options
        if option_type == "call":
            V[:, t] = np.maximum(V[:, t], S - strike)
        elif option_type == "put":
            V[:, t] = np.maximum(V[:, t], strike - S)

    # Interpolate to find the price for the given spot
    price = np.interp(spot, S, V[:, 0])
    return price

# Main Pricing Function
def price_american_option(ticker, start_date, end_date, strike, option_type="call"):
    spy_data = fetch_data(ticker, start_date, end_date)
    spot = spy_data["Close"].iloc[-1]
    rate = 0.05  # Risk-free rate
    volatility = spy_data["Close"].pct_change().std() * np.sqrt(252)
    T = 1 / 12  # 1 month
    steps = 30  # Daily steps
    samples = 10000  # QMC samples
    grid_size = 100  # FDM grid size

    price_qmc = least_squares_monte_carlo(spot, strike, rate, volatility, T, steps, samples, option_type)
    price_fdm = finite_difference_method(spot, strike, rate, volatility, T, steps, grid_size, option_type)

    return price_qmc, price_fdm, spot

# Compare Model with Historical Data
if __name__ == "__main__":
    today = datetime.date.today()
    start_date = today - datetime.timedelta(days=365)
    end_date = today

    strike = 450  # Example strike price
    option_type = "call"
    try:
        price_qmc, price_fdm, spot = price_american_option("SPY", start_date, end_date, strike, option_type)
        print(f"Spot Price: {spot:.2f}")
        print(f"Quasi-Monte Carlo American Option Price: {price_qmc:.2f}")
        print(f"Finite Difference Method American Option Price: {price_fdm:.2f}")
    except Exception as e:
        print("Error:", e)


FileNotFoundError: [Errno 2] No such file or directory: '/content/yahoo_options-2.csv'

In [ ]:
file_path = "/content/yahoo_options-2.csv"
data =  pd.read_csv(file_path)
# Dynamic Delta Hedging Simulation
def dynamic_delta_hedging(stock_prices, strike, T, r, sigma, option_type="call"):
    dt = T / len(stock_prices)
    cash_position = 0
    hedge_shares = 0
    hedging_costs = []
    pnl = []

    for t, spot_price in enumerate(stock_prices):
        time_to_maturity = T - t * dt if t < len(stock_prices) - 1 else 0

        # Compute the delta
        delta = black_scholes_delta(spot_price, strike, time_to_maturity, r, sigma, option_type)

        # Adjust hedge position
        new_hedge_shares = delta
        adjustment = new_hedge_shares - hedge_shares
        cash_position -= adjustment * spot_price
        hedge_shares = new_hedge_shares

        # Calculate P&L
        if t > 0:
            hedging_pnl = hedge_shares * spot_price + cash_position
            pnl.append(hedging_pnl)
        else:
            pnl.append(0)

        hedging_costs.append(adjustment * spot_price)

    final_pnl = hedge_shares * stock_prices[-1] + cash_position
    return pnl, final_pnl, hedging_costs